# Fine-tune PILOT for OCR on HierText

This notebook learns `<ocr>` recognition on **ground-truth line crops** from HierText.
Lines stay within their original official image split, so two crops from one image cannot
leak between train, validation and test.

Why lines? The official annotations guarantee word order inside a line, but not line order
inside a paragraph. A full-page transcript assembled by blindly concatenating lines would
create ambiguous supervision. Crops give a clear first OCR fine-tuning example.
This is a **recognition experiment**, not the official end-to-end detection/recognition benchmark.

Install the dependencies in the notebook kernel environment and download the released PILOT
checkpoint/tokenizer before the training section:

~~~bash
pip install -e ".[notebooks]"
jupyter lab
~~~

Prepared data, configuration, metrics and checkpoints are local files.

**Example settings, not tuned hyperparameters.** These notebooks illustrate a workflow; image
resolution, crop height, learning rate, noise, effective batch size, sequence limit and epoch count
have not been optimized for this dataset. Compare settings on validation and reserve test for final
evaluation. The small default subsets check execution and do not establish model quality.

A few hundred training examples are generally insufficient for robust adaptation: the model may
memorize examples, generate unstable outputs or fail on new layouts and text. Several thousand
diverse labeled examples are often a more useful starting point, but there is no universal sample
threshold or guarantee; annotation quality, task complexity and similarity to pretraining matter.
Use learning curves to decide whether more data are needed. Synthetic data can broaden coverage,
provided they match the intended domain; evaluate on separate, representative real documents.

**Resolution affects recognition.** Excessive downscaling can remove strokes and make characters
indistinguishable; later upscaling cannot recover that information. Inspect the actual prepared images,
especially small text. Higher resolution increases encoder cost, memory and visual sequence length.
Keep the same resize/crop/padding/normalization policy at inference and tune this tradeoff on validation.

## 0. Setup

In [ ]:
from pathlib import Path
import gc
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pilot").is_dir() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Place this notebook inside the PILOT repository and launch Jupyter from that repository.")
sys.path.insert(0, str(ROOT))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
import torch

from examples.finetuning.common import audit_targets, experiment_model_config, read_jsonl, save_json
from examples.finetuning.evaluation import history_table, load_best, predict_records
from pilot.training.trainer import train

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
from examples.finetuning.hiertext import prepare_hiertext
from pilot.training.metrics import sequence_metrics


In [ ]:
DATA_DIR = ROOT / "data" / "hiertext"
RAW_DIR = DATA_DIR / "raw"
PREPARED_DIR = DATA_DIR / "prepared_lines_example"
EXPERIMENT_DIR = ROOT / "runs" / "hiertext_lines_example"
MODEL_CONFIG = ROOT / "configs" / "pilot_generic.json"
IMAGE_LIMITS = {"train": 50, "validation": 10, "test": 10}
MAX_LINES_PER_IMAGE = 8
LINE_HEIGHT = 64
MAX_SIDE = 1280
EPOCHS = 10
EXTRA_TOKENS = []
SELECTION_METRIC = "cer"
METRIC_FIELDS = []


## 1. Prepare data once

Source: [official HierText repository](https://github.com/google-research-datasets/hiertext),
CC BY-SA 4.0. Ground-truth files are pinned to a Git revision. They total about 83 MB compressed.
Original image archives are much larger: the helper streams only an initial archive prefix
until `IMAGE_LIMITS` source images have been cached, then closes the connection.
It does not download the entire ~2.8 GB train image archive for this small example.

Sampling uses archive order, not a uniformly random sample of the whole corpus.
The selected source IDs are saved for reproducibility. Set a split limit to `None` to use
its complete source-image archive when scaling up.

We keep legible nonvertical lines of 2–160 characters, up to the configured number per image.
Each line is cropped by the enclosing rectangle of its official polygon with a small margin,
converted to RGB, scaled toward `LINE_HEIGHT` (64 pixels here) without exceeding the maximum side length,
then padded to a minimum dimension of 64 pixels.
The prompt is `<ocr>` and the target is the line text with normalized whitespace.

In [ ]:
prepared = prepare_hiertext(RAW_DIR, PREPARED_DIR, image_limits=IMAGE_LIMITS,
                           max_lines=MAX_LINES_PER_IMAGE, max_side=MAX_SIDE, line_height=LINE_HEIGHT)
display(pd.DataFrame({"line_examples": prepared["counts"], "source_images": prepared["source_counts"]}))


In [ ]:
examples = read_jsonl(PREPARED_DIR / "train.jsonl")
for example in examples[:3]:
    with Image.open(PREPARED_DIR / example["image"]) as image:
        preview = image.copy()
    preview.thumbnail((1000, 200))
    display(preview)
    print("Source:", example["source_id"], "| Prompt:", example["prompt"], "| Target:", example["target"])


Prepared crops/manifests and training-only RGB statistics are cached.
To train after restarting the kernel without running preparation again:

~~~python
prepared = json.loads((PREPARED_DIR / "prepared.json").read_text())
~~~

A normal rerun also reuses the cache. Changing limits, crop count or image size requires a new
`PREPARED_DIR`. The saved `source_id` preserves the original image identity for split checks.

## 2. Audit and train

The base vocabulary already contains `<ocr>`. No new task/field tokens are added.
Statistics come only from the exact prepared training pixels,
and the target-length audit includes BOS, prompt, text and EOS.

In [ ]:
model_config_path = experiment_model_config(MODEL_CONFIG, EXPERIMENT_DIR / "config" / "model.json", prepared["image_stats"])
length_summary, max_length = audit_targets(PREPARED_DIR, model_config_path, EXTRA_TOKENS)
display(pd.DataFrame(length_summary))
print("Dataset max_length:", max_length)
print("Decoder positional capacity is unchanged.")


Use batch size 1 for variable-width line images and accumulation for a larger effective batch.
The best model minimizes **generated validation CER**, computed after greedy decoding.
Generated WER/exact match and training loss are also logged to local JSONL.
No validation loss is computed. Noise rate 0.05 is an illustrative setting, not a claimed optimum.

Training uses a simple cross-entropy averaged over supervised target tokens, including structural
tokens and EOS; prompt and padding positions are masked. Frequent or long token categories can
dominate that average. A more elaborate objective could balance lexical, field-tag, coordinate,
separator and EOS contributions or normalize separate loss terms. The categories, weights and any
benefit are dataset-dependent: this example leaves them unweighted and makes no optimality claim.
Evaluate changes using generated CER or field F1, not training loss alone.

In [ ]:
RUN_DIR = EXPERIMENT_DIR / "training"
training_config = {
    "model_config": str(model_config_path),
    "train_manifest": str(PREPARED_DIR / "train.jsonl"),
    "validation_manifest": str(PREPARED_DIR / "validation.jsonl"),
    "output_dir": str(RUN_DIR),
    "extra_tokens": EXTRA_TOKENS,
    "epochs": EPOCHS,
    "learning_rate": 1e-5,
    "batch_size": 1,
    "accumulation_steps": 4,
    "max_length": max_length,
    "num_workers": 0,
    "precision": "auto",
    "gradient_checkpointing": False,
    "noise_rate": 0.05,
    "noise_strategy": "sequence",
    "selection_metric": SELECTION_METRIC,
    "metric_fields": METRIC_FIELDS,
    "seed": 42,
}
CONFIG_PATH = EXPERIMENT_DIR / "config" / "training.json"
save_json(CONFIG_PATH, training_config)
display(pd.Series(training_config))


### Run or resume

The cell resumes a completed-epoch checkpoint automatically when one exists.
Keep settings and epoch budget unchanged when resuming, or choose a new `EXPERIMENT_DIR`.

In [ ]:
resume = (RUN_DIR / "last.pt").is_file()
result = train(CONFIG_PATH, resume=resume)
print(result)


In [ ]:
history = history_table(RUN_DIR)
display(history)
history.plot(x="epoch", y="train_loss", marker="o", ylabel="Training token cross-entropy", grid=True)
plt.show()
history.plot(x="epoch", y=f"validation.{SELECTION_METRIC}", marker="o", ylabel=SELECTION_METRIC, grid=True)
plt.show()


In [ ]:
history.plot(x="epoch", y=["validation.cer", "validation.wer"], marker="o", ylabel="Error rate", grid=True)
plt.show()


## 3. Reload and predict held-out lines

Test crops come only from the official test source images.
The model sees pixels and a prompt; the target is used only to compute the metrics afterwards.

In [ ]:
model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
model, tokenizer, inference_config = load_best(RUN_DIR / "best" / "config.json")
predictions = predict_records(model, tokenizer, inference_config, PREPARED_DIR / "test.jsonl", max_length=max_length, limit=None)
save_json(RUN_DIR / "test_predictions.json", predictions)
print("Test examples:", len(predictions))
print("EOS termination rate:", sum(p["terminated"] for p in predictions) / len(predictions))


In [ ]:
scores = sequence_metrics(predictions)
scores["eos_rate"] = sum(row["terminated"] for row in predictions) / len(predictions)
display(pd.Series(scores))
save_json(RUN_DIR / "test_metrics.json", scores)


CER/WER are corpus edit distances divided by ground-truth character/word counts.
Case and punctuation are preserved. Targets have already undergone whitespace normalization
during preparation. Error rates are fractions and may exceed 1.
These are metrics on the tutorial's selected line crops, not official HierText end-to-end scores.

In [ ]:
for example in predictions[:3]:
    with Image.open(PREPARED_DIR / example["image"]) as image:
        preview = image.copy()
    preview.thumbnail((1000, 200))
    display(preview)
    print("Ground truth:", example["target"])
    print("Prediction:  ", example["prediction"])
    print("Reached EOS:", example["terminated"])


### Recognize a new line image

Pass an image containing the line to read. The helper applies the same preparation as training.

In [ ]:
CUSTOM_PROMPT = "<ocr>"
CUSTOM_HEIGHT = LINE_HEIGHT

def predict_image(image_path):
    from hashlib import sha256
    from examples.finetuning.common import save_rgb
    image_path = Path(image_path)
    fingerprint = sha256(image_path.read_bytes()).hexdigest()[:12]
    prepared_image = EXPERIMENT_DIR / "custom" / f"{image_path.stem}_{fingerprint}.png"
    if not prepared_image.exists():
        with Image.open(image_path) as image:
            save_rgb(image, prepared_image, max_side=MAX_SIDE, target_height=CUSTOM_HEIGHT)
    manifest = EXPERIMENT_DIR / "custom" / "example.jsonl"
    manifest.write_text(json.dumps({"image": prepared_image.name, "prompt": CUSTOM_PROMPT, "target": ""}) + "\n")
    return predict_records(model, tokenizer, inference_config, manifest, max_length=max_length, limit=1)[0]["prediction"]

# prediction = predict_image("/path/to/your/image.png")
# print(prediction)


## 4. CPU inference after training

CPU cost can be reduced through several complementary optimizations: per-channel GroupNorm
(already used by this implementation), separate self/cross-attention KV caches, channels-last encoder
layout, cached positional encodings, CPU thread tuning, and optional encoder compilation.
Combined optimizations can sometimes make inference **around 3× faster**, but this is an indicative
possibility, not a measured result or a guarantee for this notebook. Hardware, software versions,
image resolution and generated length determine the gain; some settings can be slower.

The short example below measures a baseline and an optimized run with identical inputs, decoding
settings and CPU thread count. It compares token IDs as well as latency. One example is only a local
sanity check: benchmark representative sizes and outputs, report startup separately, and recompute
CER/F1 on a held-out set before deployment. Small floating-point differences can change generation.
The benchmark uses validation images without consulting their labels and excludes disk I/O.

Compilation is optional and disabled by default. Set `COMPILE_ENCODER=True` to try
`torch.compile` on a machine with a supported compiler; its first call can take minutes and new
shapes may trigger additional compilation. Warm-up is excluded from the displayed latency.
The helper restores the eager encoder if benchmarking fails and surfaces the error.
Quantization, batching and alternative inference engines are additional experiments; recheck quality.

Only the original **`best/model.pt`**, its config and tokenizer are saved. Apply runtime optimizations
after loading these artifacts; do not export a compiled module as the training checkpoint.
See the [performance guide](../docs/performance.md) for scope and upstream references.

In [ ]:
from examples.finetuning.cpu import available_cpus, benchmark_cpu

CPU_THREADS = min(4, available_cpus())
COMPILE_ENCODER = False
CPU_EXAMPLES = 1
CPU_REPEATS = 3
cpu_report = benchmark_cpu(
    model, tokenizer, inference_config, PREPARED_DIR / "validation.jsonl",
    max_length=max_length, num_threads=CPU_THREADS, compile_encoder=COMPILE_ENCODER,
    examples=CPU_EXAMPLES, repeats=CPU_REPEATS,
)
display(pd.DataFrame(cpu_report["per_example"]))
print(f"Measured speedup: {cpu_report['speedup']:.2f}x | Identical token IDs: {cpu_report['same_token_ids']}")
if not cpu_report["same_token_ids"]:
    print("Outputs differ: recompute CER/F1 before interpreting this as an equivalent-quality speedup.")
save_json(RUN_DIR / "cpu_benchmark.json", cpu_report)
torch.set_num_threads(CPU_THREADS)


## Extending this example

To learn full-page OCR with boxes, build a separate manifest with a documented reading-order
policy and coordinate transformation. Do not reuse this crop-recognition score as a page-level score.
Keep all derived examples grouped by original source image.

Citation: Shangbang Long et al., *Towards End-to-End Unified Scene Text Detection and Layout Analysis*,
CVPR 2022. See the official repository for the polygon-based end-to-end evaluation protocol.